# Demo: Cylinder MASCAF

Fit a cable morphology to the cylinder demo mesh. The skeleton stops short of the ends, so the fit is extended at each tip and then scaled to the mesh surface area.

In [26]:
from mascaf import *
import logging

logging.basicConfig(level=logging.INFO)

## Mesh and skeleton

Load the demo mesh and its curve skeleton. The skeleton is the centerline the cable fit follows.

In [36]:
mm = MeshManager(mesh_path="../../data/demo/cylinder.obj")
mm.print_mesh_analysis()
skeleton = SkeletonGraph.from_txt("../../data/demo/cylinder.polylines.txt")
mesh_fig, camera = visualize_mesh_3d(
    mesh=mm,
    title="Cylinder",
    skel=skeleton,
    show_axes=False,
    width=800,
    height=600,
    eye_scale=2.2,
    return_camera=True,
)
mesh_fig

INFO:mascaf.mesh:Loaded mesh: 34 vertices, 64 faces
INFO:mascaf.mesh:Mesh Analysis Report
INFO:mascaf.mesh:====================
INFO:mascaf.mesh:
Geometry:
INFO:mascaf.mesh:  * Vertices: 34
INFO:mascaf.mesh:  * Faces: 64
INFO:mascaf.mesh:  * Components: 1
INFO:mascaf.mesh:  * Volume: 30.61
INFO:mascaf.mesh:  * Bounds: [-1.0, -1.0, -5.0] to [1.0, 1.0, 5.0]
INFO:mascaf.mesh:
Mesh Quality:
INFO:mascaf.mesh:  * Watertight: True
INFO:mascaf.mesh:  * Winding Consistent: True
INFO:mascaf.mesh:  * Normal Direction: outward
INFO:mascaf.mesh:  * Duplicate Vertices: 0
INFO:mascaf.mesh:  * Degenerate Faces: 0
INFO:mascaf.mesh:
Topology:
INFO:mascaf.mesh:  * Genus: 0
INFO:mascaf.mesh:  * Euler Characteristic: 2
INFO:mascaf.mesh:
No issues detected
INFO:mascaf.mesh:
Recommendation:
INFO:mascaf.mesh:  Mesh appears to be in good condition.
INFO:mascaf.mesh:====================


## Cable fit

`CableFitter` resamples the skeleton so edges are at most `max_edge_length` and estimates a radius at each sample from the mesh cross-section.

In [28]:
max_edge_length = 1.0

morph = CableFitter(FitOptions(max_edge_length=max_edge_length)).fit(mm.mesh, skeleton)

INFO:mascaf.cable_fitting:Starting cable fit with 4 skeleton nodes, 3 skeleton edges, 34 mesh vertices, and max_edge_length=1.0
INFO:mascaf.morphology_graph:Resampled morphology: 11 nodes, 10 edges (max_edge_length=1.0000, source 4 nodes / 3 edges, 1 sections)
INFO:mascaf.cable_fitting:Computing node radii: strategy=equivalent_area, 11 nodes, probe_eps=1.00e-04, tries=3
INFO:mascaf.cable_fitting:Radii computed: min=0.987165 max=0.987165 mean=0.987165 median=0.987165
INFO:mascaf.cable_fitting:Finished cable fit with 11 morphology nodes and 10 morphology edges


## Fitted cable

The mesh and skeleton are on the left; the cable model is on the right. Both panels use the same view. Validation compares cable volume and surface area with the mesh.

In [37]:
fig = visualize_mesh_cable_3d(
    mm, skeleton, morph, show_axes=False, eye_scale=2.2, camera=camera
)
fig.show()

validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:swctools.geometry:batch_frusta count=10 sides=16 end_caps=False verts=320 faces=320
INFO:swctools.geometry:FrustaSet.from_swc_model edges=10 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=10 show_frusta=True show_centroid=True


INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 34 vertices, 64 faces
INFO:mascaf.validation:  Skeleton: 4 nodes, 3 edges
INFO:mascaf.validation:  MorphologyGraph: 11 nodes, 10 edges
INFO:mascaf.validation:No branch vertices (degree > 2) found; overlap correction has no effect. Showing account_for_overlaps=False only.
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       30.6147
INFO:mascaf.validation:---- Morphology volume: 30.0511
INFO:mascaf.validation:---- Ratio:             0.9816
INFO:mascaf.validation:---- Error:             -0.5635
INFO:mascaf.validation:---- Relative error:    -1.84%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         68.5518
INFO:mascaf.validation:---- Morphology area:   67.0066
INFO:mascaf.validation:---- Ratio:             0.9775
INFO:mascaf.validation:---

## Surface-area normalization

Scale every radius by one factor so the cable surface area matches the mesh.

In [34]:
morph_scaled = morph.copy()
morph_scaled.scale_radii_to_match_mesh(
    mm.mesh, metric="surface_area", account_for_overlaps=False
)
fig = visualize_mesh_cable_3d(
    mm, skeleton, morph_scaled, show_axes=False, eye_scale=2, camera=camera
)
fig.show()

validator = Validation(mm, skeleton, morph_scaled)
validator.full_validation()

INFO:swctools.geometry:batch_frusta count=10 sides=16 end_caps=False verts=320 faces=320
INFO:swctools.geometry:FrustaSet.from_swc_model edges=10 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=10 show_frusta=True show_centroid=True


INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 34 vertices, 64 faces
INFO:mascaf.validation:  Skeleton: 4 nodes, 3 edges
INFO:mascaf.validation:  MorphologyGraph: 11 nodes, 10 edges
INFO:mascaf.validation:No branch vertices (degree > 2) found; overlap correction has no effect. Showing account_for_overlaps=False only.
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       30.6147
INFO:mascaf.validation:---- Morphology volume: 31.3322
INFO:mascaf.validation:---- Ratio:             1.0234
INFO:mascaf.validation:---- Error:             0.7175
INFO:mascaf.validation:---- Relative error:    2.34%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         68.5518
INFO:mascaf.validation:---- Morphology area:   68.5518
INFO:mascaf.validation:---- Ratio:             1.0000
INFO:mascaf.validation:---- 